In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import seaborn as sns
import matplotlib.pyplot as plt

np.random.seed(42)
n = 500
neighborhoods = ['Downtown', 'Uptown', 'Suburb']
data = {
    'square_footage': np.random.normal(2000, 500, n).astype(int),
    'bedrooms': np.random.randint(2, 6, n),
    'bathrooms': np.random.randint(1, 4, n),
    'age': np.random.randint(0, 50, n),
    'neighborhood': np.random.choice(neighborhoods, n)
}
df = pd.DataFrame(data)
base_price = 50000
df['price'] = (df['square_footage'] * 150 +
               df['bedrooms'] * 10000 +
               df['bathrooms'] * 8000 -
               df['age'] * 1000 +
               df['neighborhood'].map({'Downtown': 50000, 'Uptown': 30000, 'Suburb': 10000}) +
               np.random.normal(0, 20000, n)).astype(int)
df.loc[np.random.choice(df.index, 10), 'bathrooms'] = np.nan
df.loc[np.random.choice(df.index, 10), 'square_footage'] = np.nan
df['bathrooms'].fillna(df['bathrooms'].median(), inplace=True)
df['square_footage'].fillna(df['square_footage'].mean(), inplace=True)
df = pd.get_dummies(df, columns=['neighborhood'], drop_first=True)
X = df.drop('price', axis=1)
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("R² Score:", r2_score(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred))
new_data = {
    'square_footage': 2400,
    'bedrooms': 3,
    'bathrooms': 2,
    'age': 10,
    'neighborhood_Uptown': 0,
    'neighborhood_Suburb': 0
}
new_df = pd.DataFrame([new_data])
new_df = new_df.reindex(columns=X.columns, fill_value=0)
predicted_price = model.predict(new_df)
print(f"Predicted Price: ${predicted_price[0]:,.2f}")


R² Score: 0.898060013618918
RMSE: 724873375.6143885
Predicted Price: $447,962.75


<ipython-input-10-ed584d3d7fdf>:29: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['bathrooms'].fillna(df['bathrooms'].median(), inplace=True)
<ipython-input-10-ed584d3d7fdf>:30: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline

np.random.seed(42)
n = 1000
spam_words = ['free', 'win', 'cash', 'click', 'buy', 'offer']
ham_words = ['meeting', 'project', 'report', 'schedule', 'team', 'update']
emails = []
labels = []
for _ in range(n):
    if np.random.rand() > 0.5:
        text = ' '.join(np.random.choice(spam_words, size=np.random.randint(5, 15)))
        labels.append(1)
    else:
        text = ' '.join(np.random.choice(ham_words, size=np.random.randint(5, 15)))
        labels.append(0)
    emails.append(text)
has_links = np.random.randint(0, 2, n)
email_length = [len(e.split()) for e in emails]
sender_addresses = ['promo@ads.com' if l == 1 else 'colleague@work.com' for l in labels]
df = pd.DataFrame({
    'content': emails,
    'has_link': has_links,
    'length': email_length,
    'sender': sender_addresses,
    'label': labels
})
le = LabelEncoder()
df['sender'] = le.fit_transform(df['sender'])
X_text = df['content']
X_other = df[['has_link', 'length', 'sender']]
y = df['label']
vectorizer = TfidfVectorizer()
X_text_vec = vectorizer.fit_transform(X_text)
from scipy.sparse import hstack
X_final = hstack([X_text_vec, X_other])
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
new_email = ['win cash offer click now']
new_has_link = [1]
new_length = [5]
new_sender = le.transform(['promo@ads.com'])
new_text_vec = vectorizer.transform(new_email)
new_other = np.array([[new_has_link[0], new_length[0], new_sender[0]]])
from scipy.sparse import hstack
new_final = hstack([new_text_vec, new_other])
prediction = model.predict(new_final)
print("Spam" if prediction[0] == 1 else "Not Spam")


Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       105
           1       1.00      1.00      1.00        95

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200

Spam


In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

np.random.seed(42)
n = 1000
spending = np.random.normal(1000, 300, n)
age = np.random.randint(18, 70, n)
visits = np.random.randint(1, 20, n)
frequency = np.random.uniform(0.5, 5.0, n)
label = ((spending > 1100) & (visits > 10) & (frequency > 2)).astype(int)
df = pd.DataFrame({
    'spending': spending,
    'age': age,
    'visits': visits,
    'frequency': frequency,
    'label': label
})
df.loc[np.random.choice(df.index, 10), 'spending'] = np.nan
df['spending'].fillna(df['spending'].mean(), inplace=True)
df = df[(df['spending'] < 2500) & (df['frequency'] <= 5)]
X = df.drop('label', axis=1)
y = df['label']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
coefficients = model.coef_[0]
intercept = model.intercept_[0]
features = X.columns.tolist()
print("\nSeparating hyperplane:")
print(" + ".join([f"{coeff:.2f}*{feat}" for coeff, feat in zip(coefficients, features)]) + f" + {intercept:.2f} = 0")
print("\nClassification Rule: If above expression > 0, predict High-Value (1), else Low-Value (0)")


Accuracy: 0.925
              precision    recall  f1-score   support

           0       0.93      0.99      0.96       169
           1       0.90      0.58      0.71        31

    accuracy                           0.93       200
   macro avg       0.91      0.78      0.83       200
weighted avg       0.92      0.93      0.92       200


Separating hyperplane:
2.08*spending + -0.05*age + 1.74*visits + 1.41*frequency + -4.29 = 0

Classification Rule: If above expression > 0, predict High-Value (1), else Low-Value (0)


<ipython-input-13-fdeda2c6a89f>:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['spending'].fillna(df['spending'].mean(), inplace=True)
